```{tip}
**For the presenter:** before starting the slideshow, run *Kernel → Restart Kernel and Run All Cells*, so all widgets are shown.
```

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
# Load the shared model setup for the column-bypass experiment.
%run ../resources/first_steps/lesson_setup.py
gui_component_system, gui_units, gui_sheet, gui_process = rtd_system_model()
# Prepare the instrument view for demonstrating the column bypass.
bypass_instrument = rtd_bypass_widget()

# 02.4 Residence Time Distributions

## Contents

**Tools:** CADET-GUI widgets: bypass diagram, unit parameters and simulation; CADET-Process (Python).

**Python used:** Object creation, parameter assignment, unit connections and simple calculations.

**Introduces:** `Cstr`, `TubularReactor`, residence-time distributions, centroids and system delay.

1. Why the instrument matters
2. The system response
3. Set up and simulate the model in Python
4. System delay, volume and column characterization
5. Additional example: numerical dispersion

## How much of the tracer delay comes from the column?

In First Model, the tracer passed through a column alone. In a laboratory instrument, it also passes through **mixers and tubing**. The detector records their combined response.

A later or broader tracer signal can therefore come from the instrument as well as the column. **A column measurement alone cannot separate these contributions.**

The first characterization measurement bypasses the column. We represent the remaining hardware with `Cstr` and `TubularReactor`, define a tracer pulse with `Process` events, and use the simulation results to quantify delay and spreading.

```{note}
A nonbinding tracer follows liquid transport. Its **residence time** is the time it spends in the system; the distribution describes how these times vary across molecules.
```

## Bypassing the column

The **column bypass** leaves the mixer and tubing in the flow path. Their contribution remains in the detector signal, while the column contribution is removed.

Clearing **Column** removes the column from the flow path, while mixer and tubing remain connected.

In [ ]:
bypass_instrument.display()

## Part 1: the system response

Our model is **Inlet → Mixer → Tubing → Outlet**. The outlet represents the detector signal.

| Setting | Value |
|---|---:|
| Flow rate | **6 mL/min** |
| Mixer liquid volume | **1 mL** |
| Tubing liquid volume | **2 mL** |
| Tracer feed | **1 mol/m³ for 5 s**, then zero |
| Measurement duration | **300 s** |

The **5 s pulse** is defined by inlet events, as in Inlet Profiles. Its response reveals how the instrument changes a known input. The mixer and tubing settings are supplied here to explain those changes; in the lab, their effective transport parameters are what we characterize.

## Mixing adds a tail

```{figure} ./resources/mixer_symbol.svg
:width: 110px
:align: center
:alt: Mixer symbol from the CADET-GUI instrument diagram.
```

The mixer is represented by a `Cstr`: incoming liquid mixes immediately with its contents. Tracer starts leaving while some remains in the mixer, producing a tail.

The mean residence time is

$$\tau = \frac{V}{Q}.$$

At **1 mL** and **6 mL/min**, the mixer adds a mean delay of **10 s**. A larger mixing volume increases both delay and tailing.

```{dropdown} Ideal mixer response
For an instantaneous tracer pulse, $E(t)=\frac{1}{\tau}e^{-t/\tau}$ for $t\geq0$. The RTD has area one. Our finite **5 s** injection also contributes to the measured width.
```

## Tubing adds transport delay

```{figure} ./resources/tubing_symbol.svg
:width: 180px
:align: center
:alt: Tubing symbol from the CADET-GUI instrument diagram.
```

The `TubularReactor` represents liquid transport through tubing. In ideal plug flow, every fluid element spends the same time in the tube: the pulse moves through unchanged.

At **2 mL** and **6 mL/min**, tubing adds a mean delay of **20 s**. Axial dispersion spreads the pulse around that delay.

| Feature of the signal | What it helps characterize |
|---|---|
| Mean delay | Liquid volume at the known flow rate |
| Width and tailing | Mixing and dispersion |

## The response of the connected system

The mixer produces the tail; tubing carries that response to the detector and adds delay. Changing one known volume shows how that part contributes to the complete signal.

The simulation panel plots the signal leaving one unit at a time; its **Show** selector chooses which: `mixer` (the mixer response), `tubing` (with the added tubing delay) or `outlet` (the detector signal). Increasing the mixer volume `init_liquid_volume` from **1e-6 to 3e-6 m³** at unchanged flow triples its mean delay and extends the tail.

In [ ]:
from cadetgui import SimulationWidget

compact_unit_widget(gui_component_system, gui_units, create=False).display()
SimulationWidget(gui_process).display()

## Part 2: set up mixer and tubing in Python

The familiar tracer, inlet, outlet and flow sheet are supplied. Adding a `Cstr` and a `TubularReactor` creates a reusable instrument model with the original **1 mL mixer** and **2 mL tubing**, independent of the widgets.

In [ ]:
# The familiar tracer, inlet and outlet from First Model, in a flow sheet without connections.
import numpy as np
from CADETProcess.processModel import (
    ComponentSystem, Inlet, Outlet, FlowSheet, Process, Cstr, TubularReactor,
)

component_system = ComponentSystem(["Tracer"])
inlet = Inlet(component_system, "inlet")
inlet.c = [0]  # The pulse is set by events.
inlet.flow_rate = 1e-7  # 6 mL/min
outlet = Outlet(component_system, "outlet")
flow_sheet = FlowSheet(component_system, "flow_sheet")
flow_sheet.add_unit(inlet)
flow_sheet.add_unit(outlet)

### Set up the mixer with `Cstr`

We create a `Cstr` named `"mixer"` with:

| Attribute | Value | Meaning |
|---|---|---|
| `init_liquid_volume` | `1e-6` m³ (1 mL) | liquid volume of the mixer |
| `const_solid_volume` | `0` | our mixer contains no solid material |

The mixer needs no flow rate: the flow sheet passes the inlet flow through it, so outflow equals inflow and its liquid volume stays constant.

In [ ]:
# Mixer setup
mixer = Cstr(component_system, "mixer")
mixer.init_liquid_volume = 1e-6  # 1 mL.
mixer.const_solid_volume = 0

### Set up the tubing with `TubularReactor`

Next comes a `TubularReactor` named `"tubing"`. Length and diameter define its liquid volume, $V=\pi d^2L/4$; for **2 mL** in a **1 m** tube, the diameter follows as $d=\sqrt{4V/(\pi L)}$, computed with `np.sqrt` and `np.pi`.

| Attribute | Value |
|---|---|
| `length` | `1` m |
| `diameter` | from $V = 2 \times 10^{-6}$ m³ |
| `axial_dispersion` | `1e-8` m²/s |

In [ ]:
# Tubing setup: a 1 m tube containing 2 mL of liquid
tubing = TubularReactor(component_system, "tubing")
tubing.length = 1
tubing.diameter = np.sqrt(4 * 2e-6 / (np.pi * tubing.length))
tubing.axial_dispersion = 1e-8  # m²/s.

### Connect the instrument units

The new units are added to the flow sheet and connected in flow order. The process applies the same **5 s** tracer pulse as the widget, recorded over **300 s**.

In [ ]:
flow_sheet.add_unit(mixer)
flow_sheet.add_unit(tubing)
flow_sheet.add_connection(inlet, mixer)
flow_sheet.add_connection(mixer, tubing)
flow_sheet.add_connection(tubing, outlet)

In [ ]:
# The 5 s tracer pulse as inlet events, as in Inlet Profiles, recorded over 300 s.
process = Process(flow_sheet, "system")
process.cycle_time = 300
process.add_event("pulse_on", "flow_sheet.inlet.c", 1, 0)
process.add_event("pulse_off", "flow_sheet.inlet.c", 0, 5)

### Simulate the connected process

`Cadet.simulate(process)` returns the recorded solutions. The supplied plotting helper overlays the mixer and detector responses; time is in **seconds**.

In [ ]:
from CADETProcess.simulator import Cadet

simulator = Cadet()
simulation_results = simulator.simulate(process)
# Show the two outlet signals using the shared plotting helper.
_ = plot_rtd_response(simulation_results)

````{tip}
Each unit's outgoing signal is available as `results.solution.<unit>.outlet`. For the detector signal of this simulation:

```python
detector = simulation_results.solution.outlet.outlet
detector.time            # time points in seconds
detector.solution[:, 0]  # tracer concentration at these times
```

The centroid calculation below uses exactly these two arrays.
````

## A pulse has more than one arrival time

The **peak time** marks the highest concentration. Molecules also arrive before and after the peak; the tail contributes to the **mean arrival time**.

The pulse centroid includes the whole signal:

$$\bar t=\frac{\int t\,c(t)\,dt}{\int c(t)\,dt}.$$

Our **5 s** injection has its centroid at **2.5 s**. The difference between detector and inlet centroids gives the instrument's mean delay:

$$\tau_\mathrm{sys}=\bar t_\mathrm{detector}-2.5~\mathrm{s}.$$

```{note}
**System delay** is the mean residence time in the mixer and tubing outside the column, also called **extra-column delay**. Subtracting the inlet centroid accounts for the finite injection duration.
```

## The bypass experiment gives the system delay

With the column bypassed, the detector centroid contains only injection, mixer and tubing contributions. Removing the injection's **2.5 s** gives the system delay. Comparing mixer and detector centroids separates the tubing contribution in this model.

`pulse_centroid()` applies the concentration-weighted mean shown in the formula. The plotting helper marks the mean arrival times on the same signals.

In [ ]:
# Remove the 5 s injection's mean arrival time from the detector centroid.
centroid = pulse_centroid(simulation_results.solution.outlet.outlet)
tau_system = centroid - 2.5
print(f"Extra-column delay: {tau_system:.1f} s")
_ = plot_rtd_response(simulation_results, mark_centroids=True)

## From system delay to liquid volume

At a known flow rate, the mean system delay gives its effective liquid volume:

$$V_\mathrm{sys}=Q\tau_\mathrm{sys}.$$

At **6 mL/min**, the measured **30 s** delay corresponds to **3 mL** of liquid in the mixer and tubing.

In [ ]:
# Convert the measured system delay to liquid volume at 6 mL/min.
volume_system = 1e-7 * tau_system
print(f"Extra-column volume: {volume_system * 1e6:.1f} mL")

```{note}
The centroid gives the **total volume**, but cannot assign it to mixer or tubing alone. The response shape provides additional information about mixing and dispersion.
```

## System delay and column characterization

With the column reconnected, a nonbinding tracer measurement includes both system and column transport:

$$\tau_\mathrm{column}=\tau_\mathrm{measured}-\tau_\mathrm{sys}, \qquad
\varepsilon_t=\frac{Q\tau_\mathrm{column}}{V_\mathrm{column}}.$$

Assigning the full measured delay to the column would overestimate its tracer-accessible porosity.

```{note}
**In the exercises:** 02.4 Exercise 2 determines the system delay from the measured bypass pulse; Exercise 3 (optional) applies this correction to a measured column breakthrough and compares the apparent and corrected porosity.
```

```{warning}
Subtracting the system delay corrects the mean retention time, but not the shape of the curve: the mixer and tubing also spread the pulse, so the measured curve is wider than the column alone would make it. To fit the column's axial dispersion, simulate mixer, tubing and column together and compare that complete model with the measured curve.
```

The system remains part of the model after the column is reconnected. Its characterized settings carry into the column fit.

## Additional example: numerical dispersion

A simulated pulse can also spread because of the spatial grid. A grid-resolution comparison reveals how much the predicted shape depends on the numerical settings.

Here, the tubing-only model has **2 mL** volume and physical dispersion **1e-8 m²/s**. Three simulations use **20, 50 and 100** axial cells, set through `discretization.ncol`.

In [ ]:
# Use a fresh tubing-only model for each grid resolution.
grid_results = {}
for ncol in [20, 50, 100]:
    _, _, _, tube_process = rtd_system_model("tubing_grid", include_mixer=False)
    tube_process.flow_sheet.tubing.discretization.ncol = ncol
    grid_results[ncol] = simulator.simulate(tube_process)

In [ ]:
# Compare grid resolutions around the pulse, using seconds.
fig, ax = plt.subplots(figsize=(8, 3))
for index, (ncol, result) in enumerate(grid_results.items()):
    signal = result.solution.outlet.outlet
    ax.plot(signal.time, signal.solution[:, 0],  # column 0: Tracer
            color=f"C{index}", linestyle=("-", "--", ":")[index],
            linewidth=2.5, label=f"{ncol} axial cells")
ax.set_xlim(10, 40)
ax.set_xlabel("Time / s")
ax.set_ylabel("Concentration / mol/m³")
ax.grid(alpha=0.25)
_ = ax.legend()
fig.tight_layout()
plt.show()

The physical settings are identical in these runs. Changes in pulse shape therefore indicate sensitivity to the numerical grid. Stable results across grid refinements support the interpretation of fitted transport parameters. [99.6 Common Problems](../../99%20Misc/99.6%20Common%20Problems/) shows how to check the grid and the output time resolution.

## Summary

| Measurement | CADET-Process model | Information carried forward |
|---|---|---|
| Column-bypass tracer | `Cstr` + `TubularReactor` | Instrument delay, mixing and dispersion |
| Column tracer | Column model + characterized system | Porosity and column transport |
| Binding components | Binding model + characterized transport | Additional retention from binding |

For this instrument, **30 s** of the measured mean delay belongs to the mixer and tubing. The complete response also spreads the signal.

This order prevents instrument delay from being assigned to column porosity, and transport effects from being assigned to binding. The characterization workbench follows the same sequence.

**Now:** **02.4 Residence Time Distributions: Exercises**: determine the system delay and correct the column porosity.

**Next lesson:** **02.5 Binding** keeps transport fixed and introduces binding.